In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

In [2]:
# load data
df_att = pd.read_csv('attendance data with features.csv')

rand_seed = 34

In [3]:
# remove unnecessary features
# duplicates, linearly dependent variables, etc.

remove = ['HGP', 'VGP',
          'HPTS', 'VPTS',
          'HGF', 'VGF',
          'HGA', 'VGA',
          'HPP%', 'VPP%', 
          'HPK%', 'HPK%',
          'HS', 'VS', 
          'HSA', 'VSA']

df_att = df_att.drop(columns = remove)

In [4]:
# A, PA, CAP
df_cap = df_att.copy().drop(columns = ['LA', 'LCAP'])

# LA, LCAP
df_lcap = df_att.copy().drop(columns = ['A', 'PA','CAP'])

X_cap_a = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_a = df_att['A']

X_cap_pa = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_pa = df_att['PA']

X_lcap = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'CAP'])
y_lcap = df_att['LA']

In [5]:
# Function to compute evaluation metrics with dependent variable transformation
def compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred):
    if dep == 'A':
        y_train_true, y_train_pred = y_train, y_train_pred
        y_test_true, y_test_pred = y_test, y_test_pred

    elif dep == 'LA':
        y_train_true, y_train_pred = np.exp(y_train), np.exp(y_train_pred)
        y_test_true, y_test_pred = np.exp(y_test), np.exp(y_test_pred)

    elif dep == 'PA':
        y_train_true, y_train_pred = y_train * X_train['CAP'], y_train_pred * X_train['CAP']
        y_test_true, y_test_pred = y_test * X_test['CAP'], y_test_pred * X_test['CAP']

    return {
        "RMSE Train": sqrt(mean_squared_error(y_train_true, y_train_pred)),
        "MAE Train": mean_absolute_error(y_train_true, y_train_pred),
        "R² Train": r2_score(y_train_true, y_train_pred),
        "RMSE Test": sqrt(mean_squared_error(y_test_true, y_test_pred)),
        "MAE Test": mean_absolute_error(y_test_true, y_test_pred),
        "R² Test": r2_score(y_test_true, y_test_pred)
    }

In [6]:
# Compute Pearson and Spearman correlation matrices
corr_matrix_pearson = df_att.drop(columns=['H', 'V']).corr(method='pearson')
corr_matrix_spearman = df_att.drop(columns=['H', 'V']).corr(method='spearman')

# Function to plot and save heatmap
def plot_heatmap(corr_matrix, title, save_path):
    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, square=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=7, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=7, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')  # Save the figure
    plt.close()  # Close the plot to free memory

# Save Pearson heatmap
plot_heatmap(corr_matrix_pearson, 'Pearson Correlation Heatmap', 'pearson_heatmap.png')

# Save Spearman heatmap
plot_heatmap(corr_matrix_spearman, 'Spearman Correlation Heatmap', 'spearman_heatmap.png')

In [7]:
# Compute the difference between Pearson and Spearman correlations
corr_diff = corr_matrix_pearson - corr_matrix_spearman

# Plot and save the heatmap of the differences
plt.figure(figsize=(15, 10))
sns.heatmap(corr_diff, cmap='coolwarm', center=0, square=True)
plt.title('Difference Between Pearson and Spearman Correlations')
plt.xticks(ticks=range(len(corr_diff.columns)), labels=corr_diff.columns, fontsize=7, rotation=90)
plt.yticks(ticks=range(len(corr_diff.index)), labels=corr_diff.index, fontsize=7, rotation=0)
plt.savefig('correlation_difference_heatmap.png', dpi=300, bbox_inches='tight')  # Save the figure
plt.close()

In [8]:
# Define dependent variable(s)
dependent_vars = ['A', 'LA', 'PA']  # Replace with actual dependent variable(s)

# Get only independent variables
independent_vars = [col for col in df_att.columns if col not in dependent_vars + ['H', 'V']]

# Compute Pearson and Spearman correlations for only the dependent variable(s)
corr_pearson_dep = df_att.drop(columns=['H', 'V']).corr(method='pearson').loc[dependent_vars, independent_vars]
corr_spearman_dep = df_att.drop(columns=['H', 'V']).corr(method='spearman').loc[dependent_vars, independent_vars]

# Function to plot heatmap for dependent variables
def plot_heatmap_dep(corr_matrix, title, save_path):
    plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, cbar=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=8, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=8, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# Save heatmaps for dependent variables
plot_heatmap_dep(corr_pearson_dep, 'Pearson Correlation (Dependent Variables)', 'pearson_dep_heatmap.png')
plot_heatmap_dep(corr_spearman_dep, 'Spearman Correlation (Dependent Variables)', 'spearman_dep_heatmap.png')

In [9]:
##### Compute the difference for only the dependent variable(s)
corr_diff_dep = corr_pearson_dep - corr_spearman_dep

# Plot and save heatmap for the difference
plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
sns.heatmap(corr_diff_dep, cmap='coolwarm', center=0, cbar=True)
plt.title('Difference Between Pearson and Spearman Correlations (Dependent Variables)')
plt.xticks(ticks=range(len(corr_diff_dep.columns)), labels=corr_diff_dep.columns, fontsize=8, rotation=90)
plt.yticks(ticks=range(len(corr_diff_dep.index)), labels=corr_diff_dep.index, fontsize=8, rotation=0)
plt.savefig('correlation_difference_dep_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

In [10]:
# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# variance inflation factor
# Select numerical columns (excluding target variable)
df_numeric = df_att.drop(columns=['H', 'V', 'A', 'LA', 'PA'])  # Exclude categorical variables

# Add a constant for intercept
X = df_numeric.copy()
X['Intercept'] = 1  # Required for VIF calculation

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Drop the intercept row for interpretation
vif_data = vif_data[vif_data["Feature"] != "Intercept"]

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

# Display results
vif_data[vif_data['VIF'] >= 10]

,Feature,VIF
0,HRk,16.490040
2,HW,inf
3,HL,inf
4,HOL,inf
5,HPTS%,68.188705
6,HSOW,13.459593
7,HSOL,14.435630
8,HSRS,4013.965945
9,HSOS,28.413536
10,HGF/G,2010.902521


In [11]:
# VIF FEATURE SELECTION

# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Function to compute VIF and iteratively remove high VIF features
def calculate_vif(df, threshold=10):
    X = df.copy()
    X['Intercept'] = 1  # Required for VIF calculation
    
    while True:
        # Compute VIF for each feature
        vif_data = pd.DataFrame()
        vif_data["Feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        
        # Drop intercept row
        vif_data = vif_data[vif_data["Feature"] != "Intercept"]
        
        # Find the feature with the highest VIF
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break  # Stop if all VIF values are below the threshold
        
        # Identify the feature to remove
        feature_to_remove = vif_data.loc[vif_data["VIF"].idxmax(), "Feature"]
        print(f"Removing {feature_to_remove} with VIF {max_vif:.2f}")
        
        # Drop the feature with the highest VIF
        X = X.drop(columns=[feature_to_remove])
    
    return X.drop(columns=['Intercept'])  # Return dataframe without high-VIF features

# Run the VIF reduction process (CAP --> drop LCAP)
df_numeric_cap = df_cap.drop(columns=['H', 'V', 'A', 'PA'])
df_numeric_lcap = df_lcap.drop(columns=['H', 'V', 'LA'])

df_reduced_cap = calculate_vif(df_numeric_cap)

# Run the VIF reduction process (LCAP --> drop CAP)
df_reduced_lcap = calculate_vif(df_numeric_lcap)

print(df_reduced_cap.columns)
print(df_reduced_lcap.columns)

selected_feats_cap_vif = df_reduced_cap.columns
selected_feats_lcap_vif = df_reduced_lcap.columns

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.78
Removing HSRS with VIF 4007.60
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 62.99
Removing VW with VIF 32.97
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.06
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.76
Removing HSRS with VIF 4007.73
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 63.01
Removing VW with VIF 32.96
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.05
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Index(['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HPP', 'HPPA',
       'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO',
       'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSO

In [12]:
# LASSO stability selection
def stability_selection_lasso(dep, X, y, alpha_range=np.logspace(-4, 1, 10), 
                              n_resampling=100, selection_threshold=0.5, name_feats=''):
    """
    Performs LASSO-based stability selection, finds the best alpha, and reports RMSE/MAE/R² for train/test (averaged over all resampling iterations).
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        alpha_range (array-like): List of alpha values to test.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be selected in.

    Returns:
        best_alpha (float): Optimal alpha with lowest RMSE.
        selected_features (list): Names of selected features using best_alpha.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """
    # Split into train/test sets (y is not scaled)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Standardize X only
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    results = []
    warning_alphas = []
    model_stats = {}

    for alpha in alpha_range:
        selection_counts = np.zeros(X.shape[1])
        
        # Initialize statistics to accumulate over iterations
        rmse_train_all = []
        rmse_test_all = []
        mae_train_all = []
        mae_test_all = []
        r2_train_all = []
        r2_test_all = []

        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            
            for _ in range(n_resampling):
                X_sample, y_sample = resample(X_train_scaled, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)
                model = Lasso(alpha=alpha)
                model.fit(X_sample, y_sample)
                selection_counts += (model.coef_ != 0)

                # Make predictions
                y_train_pred = model.predict(X_train_scaled)
                y_test_pred = model.predict(X_test_scaled)

                # Compute statistics for this resample
                stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
                
                # Collect stats for averaging
                rmse_train_all.append(stats["RMSE Train"])
                rmse_test_all.append(stats["RMSE Test"])
                mae_train_all.append(stats["MAE Train"])
                mae_test_all.append(stats["MAE Test"])
                r2_train_all.append(stats["R² Train"])
                r2_test_all.append(stats["R² Test"])

            # Average the statistics over all resampling iterations
            avg_rmse_train = np.mean(rmse_train_all)
            avg_rmse_test = np.mean(rmse_test_all)
            avg_mae_train = np.mean(mae_train_all)
            avg_mae_test = np.mean(mae_test_all)
            avg_r2_train = np.mean(r2_train_all)
            avg_r2_test = np.mean(r2_test_all)

            # Store the averaged stats for this alpha
            model_stats[alpha] = {
                "RMSE Train": avg_rmse_train,
                "RMSE Test": avg_rmse_test,
                "MAE Train": avg_mae_train,
                "MAE Test": avg_mae_test,
                "R² Train": avg_r2_train,
                "R² Test": avg_r2_test
            }

        # Compute feature selection stability
        selection_frequencies = selection_counts / n_resampling
        selected_features = np.where(selection_frequencies >= selection_threshold)[0]
        num_selected = len(selected_features)

        results.append((alpha, num_selected, selected_features))

    # Extract alphas and feature counts
    alphas, feature_counts, feature_indices_list = zip(*results)

    # Plot feature count vs. alpha
    plt.figure(figsize=(12, 6))  # Increased figure size for better spacing
    plt.subplot(1, 2, 1)
    plt.plot(alphas, feature_counts, marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('Number of Features Selected')
    plt.title('Number of Features Selected for Different Alpha Values')
    
    # Plot RMSE vs. Alpha (use averaged RMSE Test)
    plt.subplot(1, 2, 2)
    plt.plot(model_stats.keys(), [stat["RMSE Test"] for stat in model_stats.values()], marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('RMSE (Test)')
    plt.title('Test RMSE for Different Alpha Values')
    
    # Adjust layout for spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.2)  # Increase horizontal and vertical space
    
    # Save the plot with a dynamic filename
    plt.savefig(f"LASSO_stability_{dep}_{name_feats}.png", dpi=300, bbox_inches='tight')
    
    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    # Find the best alpha based on the lowest RMSE on the test set
    best_alpha = min(model_stats, key=lambda a: model_stats[a]["RMSE Test"])
    best_index = alphas.index(best_alpha)
    selected_feature_indices = feature_indices_list[best_index]
    
    # Convert indices to feature names
    selected_feature_names = X.columns[selected_feature_indices]

    print(f"Optimal alpha: {best_alpha}")
    print(f"Number of features selected: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    # Print statistics for best alpha
    best_stats = model_stats[best_alpha]
    print("\nPerformance Metrics (Train/Test):")
    for metric, value in best_stats.items():
        print(f"{metric}: {value:.4f}")

    # Print alphas that triggered convergence warnings
    if warning_alphas:
        print(f"\n⚠️ Convergence warnings occurred for alpha values: {warning_alphas}")

    return best_alpha, selected_feature_names, best_stats

In [13]:
# all features
# tends to not converge because of high degree of multicollinearity

print('Dependent variable: A')
alpha_cap_a, selected_feat_cap_a_lasso, best_stats_a = stability_selection_lasso('A', X_cap_a, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 1, 10), 
                                                                                 name_feats = 'full_list')

Dependent variable: A
Optimal alpha: 2.1544346900318834
Number of features selected: 53
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPPOA', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.1439
RMSE Test: 1311.3636
MAE Train: 919.0344
MAE Test: 917.7840
R² Train: 0.5480
R² Test: 0.5250


In [14]:
# all featuress
print('Dependent variable: PA')
alpha_cap_pa, selected_feat_cap_pa_lasso, best_stats_pa = stability_selection_lasso('PA', X_cap_pa, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -3, 10),
                                                                                    name_feats = 'full_list')

Dependent variable: PA
Optimal alpha: 0.0001291549665014884
Number of features selected: 52
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.6764
RMSE Test: 1311.0969
MAE Train: 920.2885
MAE Test: 919.1297
R² Train: 0.5476
R² Test: 0.5252


In [15]:
# all features
print('Dependent variable: LA')
alpha_lcap, selected_feat_lcap_lasso, best_stats_la = stability_selection_lasso('LA', X_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-4, -3, 10),
                                                                                name_feats = 'full_list')

Dependent variable: LA
Optimal alpha: 0.0001668100537200059
Number of features selected: 52
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'LCAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1291.1586
RMSE Test: 1314.1122
MAE Train: 940.7172
MAE Test: 939.8513
R² Train: 0.5452
R² Test: 0.5230


In [16]:
# vif features
print('Dependent variable: A')
alpha_cap_a_vif, selected_feat_cap_a_lasso_vif, best_stats_a_vif = stability_selection_lasso('A', df_reduced_cap, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 2, 10),
                                                                                 name_feats = 'vif')

print('\nDependent variable: PA')
alpha_cap_pa_vif, selected_feat_cap_pa_lasso_vif, best_stats_pa_vif = stability_selection_lasso('PA', df_reduced_cap, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -2, 10),
                                                                                    name_feats = 'vif')

print('\nDependent variable: LA')
alpha_lcap_vif, selected_feat_lcap_lasso_vif, best_stats_la_vif = stability_selection_lasso('LA', df_reduced_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-5, -2, 10),
                                                                                name_feats = 'vif')

Dependent variable: A
Optimal alpha: 21.54434690031882
Number of features selected: 22
Selected Features: ['HRk', 'HAvAge', 'HSOW', 'HSOS', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VPPOA', 'VoPIM/G', 'VSV%', 'TDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1313.6126
RMSE Test: 1329.8040
MAE Train: 937.0338
MAE Test: 936.8278
R² Train: 0.5292
R² Test: 0.5115

Dependent variable: PA
Optimal alpha: 0.00046415888336127773
Number of features selected: 40
Selected Features: ['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGF/G', 'VPPOA', 'VPK%', 'VSH', 'VSHA', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'TDAY', 'WDAY', 'ThDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1311.1967
RMSE Test: 1330.3235
MAE Train: 939.6456
MAE Test: 940.4684
R² Trai

In [17]:
def stability_selection_rf(dep, X, y, n_estimators=100, n_resampling=100, selection_threshold=0.5, importance_threshold=0.5):
    """
    Performs stability selection for feature importance using Random Forest.
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        n_estimators (int): Number of trees in Random Forest.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be important in.
        importance_threshold (float): Fraction of most important features to consider in each iteration.

    Returns:
        selected_features (list): Names of selected features.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """

    # Split into train/test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Track feature importance frequency
    feature_importance_counts = np.zeros(X.shape[1])

    # Initialize statistics tracking
    rmse_train_all, rmse_test_all = [], []
    mae_train_all, mae_test_all = [], []
    r2_train_all, r2_test_all = [], []

    for _ in range(n_resampling):
        # Bootstrap resample
        X_sample, y_sample = resample(X_train, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)

        # Train Random Forest
        rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
        rf.fit(X_sample, y_sample)

        # Determine number of features to consider based on importance_threshold
        num_top_features = max(1, int(importance_threshold * len(X.columns)))  # Ensure at least 1 feature is selected
        importance_ranking = np.argsort(rf.feature_importances_)[::-1]  # Indices of features sorted by importance
        top_features = importance_ranking[:num_top_features]  
        feature_importance_counts[top_features] += 1

        # Make predictions
        y_train_pred = rf.predict(X_train)
        y_test_pred = rf.predict(X_test)

        # Compute statistics for this resample
        stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
        
        # Collect stats for averaging
        rmse_train_all.append(stats["RMSE Train"])
        rmse_test_all.append(stats["RMSE Test"])
        mae_train_all.append(stats["MAE Train"])
        mae_test_all.append(stats["MAE Test"])
        r2_train_all.append(stats["R² Train"])
        r2_test_all.append(stats["R² Test"])

    # Compute stability selection scores
    selection_frequencies = feature_importance_counts / n_resampling
    selected_features_indices = np.where(selection_frequencies >= selection_threshold)[0]
    selected_feature_names = X.columns[selected_features_indices]

    # Compute averaged statistics
    stats = {
        "RMSE Train": np.mean(rmse_train_all),
        "RMSE Test": np.mean(rmse_test_all),
        "MAE Train": np.mean(mae_train_all),
        "MAE Test": np.mean(mae_test_all),
        "R² Train": np.mean(r2_train_all),
        "R² Test": np.mean(r2_test_all),
    }

    # Plot feature stability selection (VERTICAL bar chart)
    plt.figure(figsize=(12, 6))
    plt.bar(X.columns, selection_frequencies, color="red")
    plt.axhline(selection_threshold, color="black", linestyle="--", label="Selection Threshold")
    
    plt.xticks(rotation=90)  # Rotate x-axis labels for readability
    plt.xlabel("Feature")
    plt.ylabel("Selection Frequency")
    plt.title(f"Feature Selection Stability for {dep}")
    plt.legend()

    plt.savefig(f"RF_stability_{dep}.png", dpi=300, bbox_inches='tight')

    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    print(f"Number of selected features: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    print("\nPerformance Metrics (Train/Test):")
    for metric, value in stats.items():
        print(f"{metric}: {value:.4f}")

    return selected_feature_names, stats

In [18]:
print('Dependent variable: A')
selected_feat_cap_a_rf, best_stats_a_rf = stability_selection_rf('A', X_cap_a, y_cap_a)

Dependent variable: A
Number of selected features: 31
Selected Features: ['HRk', 'HAvAge', 'HW', 'HPTS%', 'HSRS', 'HSOS', 'HGF/G', 'HGA/G', 'HPP', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'VAvAge', 'VSOS', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'CAP']

Performance Metrics (Train/Test):
RMSE Train: 685.2220
RMSE Test: 923.0389
MAE Train: 360.0360
MAE Test: 545.4414
R² Train: 0.8719
R² Test: 0.7646


In [19]:
print('Dependent variable: PA')
selected_feat_cap_pa_rf, best_stats_pa_rf = stability_selection_rf('PA', X_cap_pa, y_cap_pa)

Dependent variable: PA
Number of selected features: 31
Selected Features: ['HRk', 'HAvAge', 'HW', 'HOL', 'HPTS%', 'HSRS', 'HSOS', 'HGF/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HSV%', 'HCAN', 'VAvAge', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'CAP']

Performance Metrics (Train/Test):
RMSE Train: 686.1153
RMSE Test: 923.5539
MAE Train: 362.5320
MAE Test: 547.3937
R² Train: 0.8715
R² Test: 0.7643


In [20]:
print('Dependent variable: LA')
selected_feat_lcap_rf, best_stats_la_rf = stability_selection_rf('LA', X_lcap, y_lcap)

Dependent variable: LA
Number of selected features: 33
Selected Features: ['HRk', 'HAvAge', 'HW', 'HOL', 'HPTS%', 'HSRS', 'HSOS', 'HGF/G', 'HGA/G', 'HPP', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'VAvAge', 'VSOS', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'LCAP']

Performance Metrics (Train/Test):
RMSE Train: 686.6303
RMSE Test: 930.0417
MAE Train: 362.7006
MAE Test: 551.1257
R² Train: 0.8714
R² Test: 0.7610


In [21]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      A   R-squared:                       0.548
Model:                            OLS   Adj. R-squared:                  0.546
Method:                 Least Squares   F-statistic:                     235.3
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:08:57   Log-Likelihood:                -88580.
No. Observations:               10327   AIC:                         1.773e+05
Df Residuals:                   10273   BIC:                         1.777e+05
Df Model:                          53                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.216e+04   3831.348      3.175      0.002    4653.995    1.97e+04
HRk          -57.4918      5.299    -10.849      0.000     -67.879     -47.105
HAvAge        80.2071     13.629      5.885      0.000      53.491     106.923
HW           -26.1536      6.846     -3.820      0.000     -39.572     -12.735
HL            77.0105      7.076     10.884      0.000      63.141      90.880
HOL           15.5797      8.214      1.897      0.058      -0.520      31.680
HSOW          57.0752      7.256      7.866      0.000      42.853      71.297
HSOL          17.7377      8.846      2.005      0.045       0.398      35.078
HSRS         942.7438    120.249      7.840      0.000     707.032    1178.455
HSOS         212.3543    398.175      0.533      0.594    -568.147     992.856
HGA/G      -1191.5418    121.975     -9.769      0.000   -1430.636    -952.448
HPP            7.5261      2.439      3.086      0.002       2.746      12.307
HPPO          -3.4210      1.374     -2.490      0.013      -6.114      -0.728
HPPA          10.7865      2.411      4.474      0.000       6.061      15.512
HPPOA         -5.5758      1.212     -4.602      0.000      -7.951      -3.201
HSH          -20.8851      4.943     -4.225      0.000     -30.575     -11.195
HSHA          25.7211      5.686      4.524      0.000      14.576      36.867
HPIM/G       130.1006     30.370      4.284      0.000      70.569     189.633
HoPIM/G      -66.6980     31.718     -2.103      0.036    -128.871      -4.525
HS%          160.5028     25.443      6.308      0.000     110.629     210.377
HSV%       -1.955e+04   2579.095     -7.581      0.000   -2.46e+04   -1.45e+04
HSO          -24.8265      7.328     -3.388      0.001     -39.191     -10.463
HCAN         176.7520     32.854      5.380      0.000     112.351     241.153
VRk          -21.9061      5.329     -4.111      0.000     -32.352     -11.460
VAvAge         0.3303     13.367      0.025      0.980     -25.872      26.532
VL            21.6151      7.078      3.054      0.002       7.742      35.489
VOL           13.5291      7.383      1.833      0.067      -0.942      28.000
VSOW          -9.3837      7.085     -1.324      0.185     -23.272       4.504
VSOL          -8.7494      8.384     -1.044      0.297     -25.184       7.685
VSOS         491.8036    399.335      1.232      0.218    -290.971    1274.578
VGA/G       -147.3298    106.462     -1.384      0.166    -356.016      61.356
VPP            1.4623      2.370      0.617      0.537      -3.184       6.108
VPPO           0.7352      1.385      0.531      0.596      -1.979       3.450
VPPA           6.7535      9.049      0.746      0.456     -10.985      24.492
VPPOA         -1.5934      2.009     -0.793      0.428      -5.531       2.344
VPK%           6.5262     21.187      0.308      0.758     -35.004      48.056
VS

In [22]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PA   R-squared:                       0.171
Model:                            OLS   Adj. R-squared:                  0.167
Method:                 Least Squares   F-statistic:                     40.79
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:08:57   Log-Likelihood:                 12649.
No. Observations:               10327   AIC:                        -2.519e+04
Df Residuals:                   10274   BIC:                        -2.481e+04
Df Model:                          52                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7592      0.201      8.756      0.000       1.365       2.153
HRk           -0.0030      0.000    -10.131      0.000      -0.004      -0.002
HAvAge         0.0040      0.001      5.307      0.000       0.003       0.005
HW            -0.0014      0.000     -3.833      0.000      -0.002      -0.001
HL             0.0041      0.000     10.432      0.000       0.003       0.005
HOL            0.0009      0.000      2.094      0.036    6.07e-05       0.002
HSOW           0.0034      0.000      8.453      0.000       0.003       0.004
HSOL           0.0008      0.000      1.675      0.094      -0.000       0.002
HSRS           0.0558      0.007      8.388      0.000       0.043       0.069
HSOS          -0.0065      0.022     -0.295      0.768      -0.050       0.037
HGA/G         -0.0610      0.007     -9.049      0.000      -0.074      -0.048
HPP            0.0004      0.000      2.866      0.004       0.000       0.001
HPPO          -0.0002   7.59e-05     -2.183      0.029      -0.000   -1.69e-05
HPPA           0.0006      0.000      4.523      0.000       0.000       0.001
HPPOA         -0.0003    6.7e-05     -4.933      0.000      -0.000      -0.000
HSH           -0.0011      0.000     -3.961      0.000      -0.002      -0.001
HSHA           0.0012      0.000      3.778      0.000       0.001       0.002
HPIM/G         0.0081      0.002      4.831      0.000       0.005       0.011
HoPIM/G       -0.0042      0.002     -2.369      0.018      -0.008      -0.001
HS%            0.0079      0.001      5.595      0.000       0.005       0.011
HSV%          -1.1608      0.143     -8.136      0.000      -1.440      -0.881
HSO           -0.0012      0.000     -2.992      0.003      -0.002      -0.000
HCAN           0.0090      0.002      4.943      0.000       0.005       0.013
VRk           -0.0013      0.000     -4.292      0.000      -0.002      -0.001
VAvAge     -5.781e-06      0.001     -0.008      0.994      -0.001       0.001
VL             0.0012      0.000      3.211      0.001       0.000       0.002
VOL            0.0008      0.000      2.006      0.045    1.88e-05       0.002
VSOW          -0.0006      0.000     -1.424      0.154      -0.001       0.000
VSOL          -0.0006      0.000     -1.306      0.192      -0.002       0.000
VSOS           0.0241      0.022      1.103      0.270      -0.019       0.067
VGA/G         -0.0071      0.006     -1.200      0.230      -0.019       0.004
VPP         7.466e-05      0.000      0.571      0.568      -0.000       0.000
VPPO        2.536e-05   7.39e-05      0.343      0.732      -0.000       0.000
VPPA        5.527e-05      0.000      0.195      0.845      -0.001       0.001
VPK%          -0.0003      0.001     -0.476      0.634      -0.002       0.001
VSH            0.0004      0.000      1.646      0.100    -8.4e-05       0.001
VS

In [23]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     LA   R-squared:                       0.522
Model:                            OLS   Adj. R-squared:                  0.520
Method:                 Least Squares   F-statistic:                     215.9
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:08:57   Log-Likelihood:                 11593.
No. Observations:               10327   AIC:                        -2.308e+04
Df Residuals:                   10274   BIC:                        -2.270e+04
Df Model:                          52                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0075      0.245     -0.031      0.975      -0.488       0.473
HRk           -0.0032      0.000     -9.783      0.000      -0.004      -0.003
HAvAge         0.0045      0.001      5.335      0.000       0.003       0.006
HW            -0.0015      0.000     -3.688      0.000      -0.002      -0.001
HL             0.0044      0.000     10.242      0.000       0.004       0.005
HOL            0.0011      0.001      2.101      0.036    7.07e-05       0.002
HSOW           0.0034      0.000      7.722      0.000       0.003       0.004
HSOL           0.0010      0.001      1.797      0.072   -8.86e-05       0.002
HSRS           0.0592      0.007      8.045      0.000       0.045       0.074
HSOS          -0.0151      0.024     -0.618      0.537      -0.063       0.033
HGA/G         -0.0687      0.007     -9.198      0.000      -0.083      -0.054
HPP            0.0005      0.000      3.072      0.002       0.000       0.001
HPPO          -0.0002    8.4e-05     -2.460      0.014      -0.000    -4.2e-05
HPPA           0.0007      0.000      4.576      0.000       0.000       0.001
HPPOA         -0.0003   7.42e-05     -4.524      0.000      -0.000      -0.000
HSH           -0.0011      0.000     -3.748      0.000      -0.002      -0.001
HSHA           0.0014      0.000      4.011      0.000       0.001       0.002
HPIM/G         0.0088      0.002      4.708      0.000       0.005       0.012
HoPIM/G       -0.0043      0.002     -2.204      0.028      -0.008      -0.000
HS%            0.0089      0.002      5.729      0.000       0.006       0.012
HSV%          -1.2855      0.158     -8.132      0.000      -1.595      -0.976
HSO           -0.0013      0.000     -2.817      0.005      -0.002      -0.000
HCAN           0.0118      0.002      5.871      0.000       0.008       0.016
VRk           -0.0014      0.000     -4.223      0.000      -0.002      -0.001
VAvAge      9.043e-05      0.001      0.110      0.912      -0.002       0.002
VL             0.0013      0.000      3.084      0.002       0.000       0.002
VOL            0.0009      0.000      1.968      0.049    3.47e-06       0.002
VSOW          -0.0006      0.000     -1.486      0.137      -0.001       0.000
VSOL          -0.0006      0.001     -1.228      0.220      -0.002       0.000
VSOS           0.0264      0.024      1.090      0.276      -0.021       0.074
VGA/G         -0.0059      0.007     -0.906      0.365      -0.019       0.007
VPP         8.025e-05      0.000      0.554      0.580      -0.000       0.000
VPPO        1.576e-05   8.19e-05      0.193      0.847      -0.000       0.000
VPPA        6.312e-05      0.000      0.201      0.841      -0.001       0.001
VPK%          -0.0003      0.001     -0.437      0.662      -0.002       0.001
VSH            0.0005      0.000      1.673      0.094   -8.53e-05       0.001
VS

In [24]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      A   R-squared:                       0.533
Model:                            OLS   Adj. R-squared:                  0.531
Method:                 Least Squares   F-statistic:                     378.3
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:08:57   Log-Likelihood:                -88757.
No. Observations:               10327   AIC:                         1.776e+05
Df Residuals:                   10295   BIC:                         1.778e+05
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.205e+04   3658.633      3.294      0.001    4881.353    1.92e+04
HRk          -46.1787      5.733     -8.055      0.000     -57.416     -34.942
HAvAge        58.3369     13.644      4.276      0.000      31.592      85.082
HW            -7.6709      5.696     -1.347      0.178     -18.836       3.495
HPTS%      -5399.4062    775.946     -6.958      0.000   -6920.412   -3878.401
HSRS        2840.1317    404.849      7.015      0.000    2046.550    3633.713
HSOS       -3263.2116    541.795     -6.023      0.000   -4325.235   -2201.188
HGF/G      -2186.5483    391.741     -5.582      0.000   -2954.438   -1418.659
HGA/G       1085.7085    381.603      2.845      0.004     337.693    1833.724
HPP            2.2615      2.263      1.000      0.318      -2.174       6.696
HPPA          14.0499      2.432      5.778      0.000       9.284      18.816
HPPOA         -2.8423      0.991     -2.869      0.004      -4.784      -0.900
HSH          -14.3266      4.962     -2.887      0.004     -24.054      -4.600
HPIM/G        75.6143     22.720      3.328      0.001      31.080     120.149
HoPIM/G      -37.2981     21.631     -1.724      0.085     -79.698       5.102
HS%          172.4955     25.155      6.857      0.000     123.187     221.804
HSV%       -1.613e+04   2583.836     -6.242      0.000   -2.12e+04   -1.11e+04
VAvAge        -3.1666     13.311     -0.238      0.812     -29.259      22.926
VSOS         121.2005    396.981      0.305      0.760    -656.960     899.361
VGF/G        -20.4567     81.359     -0.251      0.801    -179.935     139.022
VGA/G        -85.4744     78.819     -1.084      0.278    -239.974      69.026
VPP           -0.7360      2.305     -0.319      0.749      -5.254       3.782
VPPO           3.8290      1.301      2.943      0.003       1.278       6.380
VPPOA          1.8682      1.122      1.665      0.096      -0.331       4.067
VPK%          -5.6528      5.499     -1.028      0.304     -16.432       5.126
VSH            7.6697      4.872      1.574      0.115      -1.881      17.221
VPIM/G        -5.2061     29.997     -0.174      0.862     -64.006      53.594
VoPIM/G       -6.1821     31.185     -0.198      0.843     -67.310      54.946
VS%           10.7817     25.388      0.425      0.671     -38.983      60.546
VSV%        4061.0787   2567.052      1.582      0.114    -970.842    9092.999
SDAY         313.7743     30.137     10.412      0.000     254.699     372.849
CAP            1.0714      0.011     96.869      0.000       1.050       1.093
==============================================================================
Omnibus:                     2239.869   Durbin-Watson:                   1.974
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4722.742
Skew:                          -1.270   Prob(JB):                         0.00
Ku

In [25]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PA   R-squared:                       0.142
Model:                            OLS   Adj. R-squared:                  0.139
Method:                 Least Squares   F-statistic:                     54.81
Date:                Fri, 04 Apr 2025   Prob (F-statistic):          3.36e-312
Time:                        00:08:58   Log-Likelihood:                 12468.
No. Observations:               10327   AIC:                        -2.487e+04
Df Residuals:                   10295   BIC:                        -2.464e+04
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7587      0.204      8.625      0.000       1.359       2.158
HRk           -0.0026      0.000     -8.173      0.000      -0.003      -0.002
HAvAge         0.0029      0.001      3.868      0.000       0.001       0.004
HW           5.13e-06      0.000      0.014      0.989      -0.001       0.001
HOL         9.807e-05      0.000      0.307      0.759      -0.001       0.001
HPTS%         -0.2973      0.046     -6.471      0.000      -0.387      -0.207
HSRS           0.0900      0.007     12.959      0.000       0.076       0.104
HSOS          -0.1203      0.022     -5.408      0.000      -0.164      -0.077
HGF/G         -0.0382      0.005     -7.036      0.000      -0.049      -0.028
HPP            0.0002      0.000      1.257      0.209   -9.17e-05       0.000
HPPO       -3.467e-05   7.31e-05     -0.474      0.635      -0.000       0.000
HPPA           0.0008      0.000      6.289      0.000       0.001       0.001
HPPOA         -0.0002   6.77e-05     -3.360      0.001      -0.000   -9.47e-05
HSH           -0.0010      0.000     -3.651      0.000      -0.002      -0.000
HPIM/G         0.0051      0.002      2.990      0.003       0.002       0.008
HoPIM/G       -0.0019      0.002     -1.069      0.285      -0.005       0.002
HSV%          -0.9464      0.144     -6.583      0.000      -1.228      -0.665
HCAN           0.0089      0.002      4.838      0.000       0.005       0.012
VAvAge        -0.0002      0.001     -0.286      0.775      -0.002       0.001
VGF/G         -0.0014      0.004     -0.310      0.757      -0.010       0.007
VGA/G         -0.0050      0.004     -1.183      0.237      -0.013       0.003
VPP        -5.628e-05      0.000     -0.442      0.659      -0.000       0.000
VPPO           0.0002    7.3e-05      2.832      0.005    6.37e-05       0.000
VPPOA          0.0001   6.24e-05      1.624      0.104    -2.1e-05       0.000
VPK%          -0.0003      0.000     -0.890      0.374      -0.001       0.000
VSH            0.0004      0.000      1.644      0.100   -8.55e-05       0.001
VPIM/G        -0.0003      0.002     -0.183      0.855      -0.004       0.003
VoPIM/G       -0.0003      0.002     -0.169      0.866      -0.004       0.003
VS%            0.0006      0.001      0.413      0.680      -0.002       0.003
VSV%           0.1920      0.142      1.352      0.176      -0.086       0.470
SDAY           0.0174      0.002     10.454      0.000       0.014       0.021
CAP         4.645e-06   6.25e-07      7.433      0.000    3.42e-06    5.87e-06
==============================================================================
Omnibus:                     2090.496   Durbin-Watson:                   1.986
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4265.228
Skew:                          -1.204   Prob(JB):                         0.00
Ku

In [26]:
#### OLS WITH LASSO SELECTED FEATS

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     LA   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.505
Method:                 Least Squares   F-statistic:                     320.6
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:08:58   Log-Likelihood:                 11431.
No. Observations:               10327   AIC:                        -2.279e+04
Df Residuals:                   10293   BIC:                        -2.255e+04
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1707      0.260     -0.658      0.511      -0.679       0.338
HRk           -0.0026      0.000     -7.326      0.000      -0.003      -0.002
HAvAge         0.0030      0.001      3.581      0.000       0.001       0.005
HW            -0.0001      0.000     -0.348      0.728      -0.001       0.001
HOL            0.0009      0.000      2.262      0.024       0.000       0.002
HPTS%         -0.3505      0.051     -6.850      0.000      -0.451      -0.250
HSRS           0.1989      0.028      7.171      0.000       0.145       0.253
HSOS          -0.2561      0.037     -6.916      0.000      -0.329      -0.184
HGF/G         -0.1546      0.027     -5.827      0.000      -0.207      -0.103
HGA/G          0.0895      0.026      3.449      0.001       0.039       0.140
HPP            0.0001      0.000      0.746      0.456      -0.000       0.000
HPPA           0.0009      0.000      5.851      0.000       0.001       0.001
HPPOA         -0.0002   6.21e-05     -3.248      0.001      -0.000   -7.99e-05
HSH           -0.0008      0.000     -2.574      0.010      -0.001      -0.000
HPIM/G         0.0062      0.001      4.425      0.000       0.003       0.009
HoPIM/G       -0.0029      0.001     -2.184      0.029      -0.005      -0.000
HS%            0.0100      0.002      6.459      0.000       0.007       0.013
HSV%          -1.1037      0.159     -6.958      0.000      -1.415      -0.793
VAvAge     -7.732e-05      0.001     -0.095      0.924      -0.002       0.002
VSOS           0.0155      0.025      0.630      0.529      -0.033       0.064
VGF/G          0.0010      0.005      0.197      0.844      -0.009       0.011
VGA/G         -0.0045      0.005     -0.925      0.355      -0.014       0.005
VPP        -2.776e-05      0.000     -0.196      0.844      -0.000       0.000
VPPO           0.0002   8.09e-05      2.246      0.025    2.31e-05       0.000
VPPA           0.0006      0.001      1.051      0.293      -0.000       0.002
VPPOA      -7.447e-07      0.000     -0.006      0.995      -0.000       0.000
VPK%           0.0010      0.001      0.763      0.446      -0.001       0.003
VSH            0.0005      0.000      1.559      0.119      -0.000       0.001
VPIM/G        -0.0006      0.002     -0.305      0.761      -0.004       0.003
VoPIM/G    -4.854e-05      0.002     -0.025      0.980      -0.004       0.004
VS%            0.0004      0.002      0.252      0.801      -0.003       0.003
VSV%           0.2435      0.158      1.539      0.124      -0.067       0.554
SDAY           0.0193      0.002     10.473      0.000       0.016       0.023
LCAP           1.1014      0.012     91.454      0.000       1.078       1.125
==============================================================================
Omnibus:                     2718.955   Durbin-Watson:                   1.982
Pr